# Lesson 4.4 - Sequence Models, Attention & Transformers

## Learning Objectives
- Build a toy bigram language model.
- Compute self-attention weights on a tiny sequence.
- Apply a reusable checklist for sequence-model application design.


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)


## Bigram Baseline: Next-Token Probabilities


In [2]:
text = 'the model learns from context and the model predicts from context'
tokens = text.split()

bigram_counts = {}
for a, b in zip(tokens[:-1], tokens[1:]):
    bigram_counts.setdefault(a, {})
    bigram_counts[a][b] = bigram_counts[a].get(b, 0) + 1

bigram_probs = {}
for token, nxt in bigram_counts.items():
    total = sum(nxt.values())
    bigram_probs[token] = {k: v / total for k, v in nxt.items()}

print('bigram_probs for "model":', bigram_probs.get('model', {}))


bigram_probs for "model": {'learns': 0.5, 'predicts': 0.5}


In [3]:
def generate(start: str, steps: int = 8) -> str:
    cur = start
    out = [cur]
    for _ in range(steps):
        dist = bigram_probs.get(cur)
        if not dist:
            break
        words, probs = zip(*dist.items())
        cur = np.random.choice(words, p=probs)
        out.append(cur)
    return ' '.join(out)

print(generate('the'))


the model predicts from context and the model predicts


## Self-Attention Toy Computation
We manually compute attention scores to connect intuition with matrix operations.


In [4]:
# 4 tokens with embedding dimension 4
X = np.array([
    [1.0, 0.2, 0.1, 0.0],
    [0.9, 0.1, 0.3, 0.2],
    [0.2, 0.8, 0.7, 0.1],
    [0.1, 0.7, 0.9, 0.3],
])

Wq = np.random.randn(4, 4) * 0.2
Wk = np.random.randn(4, 4) * 0.2
Wv = np.random.randn(4, 4) * 0.2

Q = X @ Wq
K = X @ Wk
V = X @ Wv

scores = Q @ K.T / np.sqrt(Q.shape[1])
exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
attn = exp_scores / exp_scores.sum(axis=1, keepdims=True)
out = attn @ V

print('attention matrix:')
print(pd.DataFrame(attn).round(3))
print('output shape:', out.shape)


attention matrix:
       0      1      2      3
0  0.250  0.251  0.249  0.250
1  0.252  0.251  0.249  0.248
2  0.247  0.248  0.252  0.253
3  0.249  0.249  0.251  0.251
output shape: (4, 4)


## Sequence Model Design Template
```text
Task type: classification | generation | retrieval-augmented generation
Context length requirement:
Latency budget:
Model family candidate:
Fallback strategy:
Evaluation slices (long context, edge cases, safety):
```

## Checklist
- [ ] Baseline model exists (e.g., n-gram or small encoder).
- [ ] Context truncation policy is defined.
- [ ] Cost/latency targets are explicit.
- [ ] Evaluation includes failure and edge prompts.
- [ ] Safety/grounding constraints are documented.


## Case Studies & Exceptions
### Case 1: Long Ticket Summarization Failure
A support summarizer dropped key details due to aggressive truncation. Fix: chunking + retrieval ranking + citation-aware prompts.

### Case 2: Transformer Overkill for Short Sequences
A short text classification task saw negligible gains over simpler models but much higher inference cost.

### Exception
For strict low-latency and small-context tasks, lightweight recurrent or classical models can be better operational choices.


## Interview Questions & Answers
1. **Q:** Why start with a bigram baseline?  
   **A:** It provides a quick sanity check before complex sequence models.
2. **Q:** What does self-attention compute?  
   **A:** Weighted token interactions based on query-key similarity.
3. **Q:** Why divide scores by sqrt(d)?  
   **A:** To stabilize softmax scaling at larger dimensions.
4. **Q:** Why do transformers need positional signals?  
   **A:** Attention alone is order-agnostic.
5. **Q:** When are transformers not necessary?  
   **A:** Small-context tasks with tight latency/cost constraints.
6. **Q:** What is one common LLM failure mode in production?  
   **A:** Context overflow causing missing critical facts.
7. **Q:** How do you mitigate long-context failures?  
   **A:** Retrieval + summarization + context budgeting policies.
8. **Q:** What should be monitored in sequence apps?  
   **A:** Latency, error rates, quality metrics, and drift in input patterns.
9. **Q:** Why keep fallback strategies?  
   **A:** To preserve reliability when generation quality or latency degrades.
10. **Q:** What is the practical bridge to LLM apps?  
   **A:** The same sequence and attention principles govern prompt/context design.
